In [2]:
import os
import polars as pl

DATA_PATH = "../data/raw/data.csv"

print("Dataset path :", DATA_PATH)
print("File exists  :", os.path.exists(DATA_PATH))

Dataset path : ../data/raw/data.csv
File exists  : True


In [3]:
file_size_bytes = os.path.getsize(DATA_PATH)
file_size_mb = file_size_bytes / (1024 ** 2)

print(f"Ukuran file: {file_size_mb:.2f} MB")

Ukuran file: 682.73 MB


In [4]:
lf = pl.scan_csv(
    DATA_PATH,
    infer_schema_length=1000
)

print(lf.collect_schema())

Schema({'Label': String, 'Denomination': String, 'Location': String, 'Date': String, 'Title': String, 'Sentence_Unit': String, 'Link': String})


In [5]:
sample = lf.head(10).collect()

sample

Label,Denomination,Location,Date,Title,Sentence_Unit,Link
str,str,str,str,str,str,str
"""Catholic""","""Katolisitas""","""National""",null,null,"""Banyak orang yang bertanya-tan…","""https://katolisitas.org"""
"""Catholic""","""Katolisitas""","""National""",null,null,"""Sebab ketika berselingkuh, ses…","""https://katolisitas.org"""
"""Catholic""","""Katolisitas""","""National""",null,null,"""Kalian pengen masuk Surga ga?""","""https://katolisitas.org"""
"""Catholic""","""Katolisitas""","""National""",null,null,"""Tinggallah bersama kami, ya Tu…","""https://katolisitas.org"""
"""Catholic""","""Katolisitas""","""National""",null,null,"""Perjalanan dua orang murid ke …","""https://katolisitas.org"""
"""Catholic""","""Katolisitas""","""National""",null,null,"""Kemana uang yang diberikan uma…","""https://katolisitas.org"""
"""Catholic""","""Katolisitas""","""National""",null,null,"""Karena itu penggabungan antara…","""https://katolisitas.org"""
"""Catholic""","""Katolisitas""","""National""",null,null,"""Orang tidak dapat menjadi Kato…","""https://katolisitas.org"""
"""Catholic""","""Katolisitas""","""National""",null,null,"""Bagaimana Yesus memberikan jaw…","""https://katolisitas.org"""


In [6]:
row_count = lf.select(
    pl.len().alias("jumlah_baris")
).collect()

row_count

jumlah_baris
u32
3013172


In [7]:
column_count = len(lf.collect_schema())

print(f"Jumlah kolom: {column_count}")

Jumlah kolom: 7


In [8]:
columns = lf.collect_schema().names()

print("Nama kolom:")
for i, column in enumerate(columns, start=1):
    print(f"{i}. {column}")

Nama kolom:
1. Label
2. Denomination
3. Location
4. Date
5. Title
6. Sentence_Unit
7. Link


In [9]:
schema = lf.collect_schema()

schema_df = pl.DataFrame({
    "column": schema.names(),
    "dtype": [str(dtype) for dtype in schema.values()]
})

schema_df

column,dtype
str,str
"""Label""","""String"""
"""Denomination""","""String"""
"""Location""","""String"""
"""Date""","""String"""
"""Title""","""String"""
"""Sentence_Unit""","""String"""
"""Link""","""String"""


In [10]:
null_counts = lf.select([
    pl.col(column).null_count().alias(column)
    for column in columns
]).collect(engine="streaming")

null_counts

Label,Denomination,Location,Date,Title,Sentence_Unit,Link
u32,u32,u32,u32,u32,u32,u32
0,0,0,3013172,3013172,0,0


In [11]:
null_profile = lf.select([
    pl.len().alias("total_rows"),
    *[
        pl.col(column).null_count().alias(f"{column}_null")
        for column in columns
    ]
]).collect(engine="streaming")

null_profile

total_rows,Label_null,Denomination_null,Location_null,Date_null,Title_null,Sentence_Unit_null,Link_null
u32,u32,u32,u32,u32,u32,u32,u32
3013172,0,0,0,3013172,3013172,0,0


In [14]:
null_table = pl.DataFrame({
    "column": columns,
    "null_count": [
        null_profile[f"{column}_null"][0]
        for column in columns
    ]
})

null_table = null_table.with_columns(
    (pl.col("null_count") / null_profile["total_rows"][0] * 100)
    .round(2)
    .alias("null_percentage")
)

null_table
label_counts = (
    lf
    .group_by("Label")
    .agg(
        pl.len().alias("jumlah")
    )
    .sort("jumlah", descending=True)
    .collect(engine="streaming")
)

label_counts

Label,jumlah
str,u32
"""Islam""",1455454
"""Catholic""",797131
"""Protestant""",760587


In [15]:
label_distribution = label_counts.with_columns(
    (
        pl.col("jumlah") / pl.col("jumlah").sum() * 100
    )
    .round(2)
    .alias("persentase")
)

label_distribution

Label,jumlah,persentase
str,u32,f64
"""Islam""",1455454,48.3
"""Catholic""",797131,26.45
"""Protestant""",760587,25.24


In [16]:
denomination_counts = (
    lf
    .group_by("Denomination")
    .agg(
        pl.len().alias("jumlah")
    )
    .sort("jumlah", descending=True)
    .collect(engine="streaming")
)

denomination_counts

Denomination,jumlah
str,u32
"""Muslim.or.id""",169211
"""IBTimes Indonesia""",147797
"""Majalah Narwastu""",133793
"""Katolisitas""",129996
"""KWI - Konferensi Waligereja In…",129803
…,…
"""Rubrik Kristen""",68
"""GKPPD - Gereja Kristen Protest…",25
"""ODB Ministries Indonesia""",13


In [17]:
denomination_unique = (
    lf
    .select(
        pl.col("Denomination").n_unique().alias("jumlah_kategori")
    )
    .collect(engine="streaming")
)

denomination_unique

jumlah_kategori
u32
60


In [18]:
denomination_top15 = (
    lf
    .group_by("Denomination")
    .agg(
        pl.len().alias("jumlah")
    )
    .sort("jumlah", descending=True)
    .head(15)
    .collect(engine="streaming")
)

denomination_top15

Denomination,jumlah
str,u32
"""Muslim.or.id""",169211
"""IBTimes Indonesia""",147797
"""Majalah Narwastu""",133793
"""Katolisitas""",129996
"""KWI - Konferensi Waligereja In…",129803
…,…
"""Bersama Dakwah""",104361
"""NU - Nahdlatul Ulama""",100388
"""Islami""",99668


In [19]:
label_denomination = (
    lf
    .group_by(["Label", "Denomination"])
    .agg(
        pl.len().alias("jumlah")
    )
    .sort("jumlah", descending=True)
    .head(30)
    .collect(engine="streaming")
)

label_denomination

Label,Denomination,jumlah
str,str,u32
"""Islam""","""Muslim.or.id""",169211
"""Islam""","""IBTimes Indonesia""",147797
"""Protestant""","""Majalah Narwastu""",133793
"""Catholic""","""Katolisitas""",129996
"""Catholic""","""KWI - Konferensi Waligereja In…",129803
…,…,…
"""Islam""","""Baznas - National Zakat Agency""",51367
"""Protestant""","""Reformata""",45814
"""Catholic""","""KAS - Keuskupan Agung Semarang""",45328


In [20]:
location_counts = (
    lf
    .group_by("Location")
    .agg(
        pl.len().alias("jumlah")
    )
    .sort("jumlah", descending=True)
    .collect(engine="streaming")
)

location_counts

Location,jumlah
str,u32
"""National""",2492876
"""Java""",286452
"""Sumatra""",144632
"""Sulawesi""",30024
"""Nusa Tenggara""",23684
"""Papua""",19199
"""International""",9369
"""Maluku""",5988
"""Kalimantan""",948


In [21]:
empty_string_counts = (
    lf
    .select([
        pl.col(column)
        .str.strip_chars()
        .eq("")
        .sum()
        .alias(column)
        for column in columns
    ])
    .collect(engine="streaming")
)

empty_string_counts

Label,Denomination,Location,Date,Title,Sentence_Unit,Link
u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0


In [22]:
whitespace_counts = (
    lf
    .select([
        (
            pl.col(column).is_not_null()
            & (pl.col(column) != pl.col(column).str.strip_chars())
        )
        .sum()
        .alias(column)
        for column in columns
    ])
    .collect(engine="streaming")
)

whitespace_counts

Label,Denomination,Location,Date,Title,Sentence_Unit,Link
u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,804


In [23]:
whitespace_links = (
    lf
    .filter(
        pl.col("Link").is_not_null()
        & (pl.col("Link") != pl.col("Link").str.strip_chars())
    )
    .select("Link")
    .head(10)
    .collect(engine="streaming")
)

whitespace_links

Link
str
"""https://lplhsda.mui.or.id/2023…"
"""https://lplhsda.mui.or.id/2023…"
"""https://lplhsda.mui.or.id/2023…"
"""https://lplhsda.mui.or.id/2023…"
"""https://lplhsda.mui.or.id/2023…"
"""https://lplhsda.mui.or.id/2023…"
"""https://lplhsda.mui.or.id/2023…"
"""https://lplhsda.mui.or.id/2023…"
"""https://lplhsda.mui.or.id/2023…"


In [24]:
whitespace_detail = (
    lf
    .filter(
        pl.col("Link").is_not_null()
        & (pl.col("Link") != pl.col("Link").str.strip_chars())
    )
    .select(
        [
            pl.col("Link"),
            pl.col("Link").str.len_chars().alias("panjang_asli"),
            pl.col("Link").str.strip_chars().str.len_chars().alias("panjang_setelah_strip"),
        ]
    )
    .head(10)
    .collect(engine="streaming")
)

whitespace_detail

Link,panjang_asli,panjang_setelah_strip
str,u32,u32
"""https://lplhsda.mui.or.id/2023…",59,58
"""https://lplhsda.mui.or.id/2023…",59,58
"""https://lplhsda.mui.or.id/2023…",59,58
"""https://lplhsda.mui.or.id/2023…",59,58
"""https://lplhsda.mui.or.id/2023…",59,58
"""https://lplhsda.mui.or.id/2023…",59,58
"""https://lplhsda.mui.or.id/2023…",59,58
"""https://lplhsda.mui.or.id/2023…",59,58
"""https://lplhsda.mui.or.id/2023…",59,58


In [25]:
duplicate_profile = (
    lf.select(
        [
            pl.len().alias("total_rows"),
            pl.struct(columns).n_unique().alias("unique_rows"),
        ]
    )
    .with_columns(
        (
            pl.col("total_rows") - pl.col("unique_rows")
        ).alias("duplicate_rows")
    )
    .collect(engine="streaming")
)

duplicate_profile

total_rows,unique_rows,duplicate_rows
u32,u32,u32
3013172,3012817,355


In [26]:
duplicate_percentage = (
    duplicate_profile
    .with_columns(
        (
            pl.col("duplicate_rows")
            / pl.col("total_rows")
            * 100
        )
        .round(4)
        .alias("duplicate_percentage")
    )
)

duplicate_percentage

total_rows,unique_rows,duplicate_rows,duplicate_percentage
u32,u32,u32,f64
3013172,3012817,355,0.0118


In [27]:
duplicate_sentences = (
    lf
    .group_by("Sentence_Unit")
    .agg(
        pl.len().alias("jumlah")
    )
    .filter(
        pl.col("jumlah") > 1
    )
    .sort("jumlah", descending=True)
    .head(20)
    .collect(engine="streaming")
)

duplicate_sentences

Sentence_Unit,jumlah
str,u32
"""Data Anda tidak akan pernah di…",3429
"""Digerakkan oleh insan-insan pr…",1382
"""Membantu para Waligereja mewuj…",1073
"""Kadanna Puang nakua, battakomi…",942
"""Sebenarnya Pemuda Kaya ini mem…",942
…,…
"""Dan ampunilah kesalahan kami,s…",255
"""Jadilah kehendakMu di atas bum…",255
"""Dan janganlah masukkan kami ke…",254


In [28]:
duplicate_sentence_summary = (
    lf
    .group_by("Sentence_Unit")
    .agg(
        pl.len().alias("jumlah")
    )
    .filter(
        pl.col("jumlah") > 1
    )
    .select(
        [
            pl.len().alias("jumlah_sentence_berulang"),
            (pl.col("jumlah") - 1).sum().alias("jumlah_duplicate_sentence")
        ]
    )
    .collect(engine="streaming")
)

duplicate_sentence_summary

jumlah_sentence_berulang,jumlah_duplicate_sentence
u32,u32
653620,1047021


In [29]:
text_length_stats = (
    lf
    .select(
        [
            pl.col("Sentence_Unit")
            .str.len_chars()
            .alias("char_length")
        ]
    )
    .select(
        [
            pl.len().alias("total_rows"),
            pl.col("char_length").mean().alias("mean_chars"),
            pl.col("char_length").median().alias("median_chars"),
            pl.col("char_length").min().alias("min_chars"),
            pl.col("char_length").max().alias("max_chars"),
        ]
    )
    .collect(engine="streaming")
)

text_length_stats

total_rows,mean_chars,median_chars,min_chars,max_chars
u32,f64,f64,u32,u32
3013172,122.478431,108.0,25,500


In [30]:
shortest_texts = (
    lf
    .with_columns(
        pl.col("Sentence_Unit")
        .str.len_chars()
        .alias("char_length")
    )
    .select(
        [
            "Label",
            "Denomination",
            "Location",
            "Sentence_Unit",
            "char_length",
            "Link",
        ]
    )
    .sort("char_length")
    .head(10)
    .collect(engine="streaming")
)

shortest_texts

Label,Denomination,Location,Sentence_Unit,char_length,Link
str,str,str,str,u32,str
"""Catholic""","""Hidup Katolik Magazine""","""National""","""Ya, itu kan gaya menulis.""",25,"""https://www.hidupkatolik.com/2…"
"""Catholic""","""Hidup Katolik Magazine""","""National""","""Dadan … ini hujan rezeki.""",25,"""https://www.hidupkatolik.com/2…"
"""Catholic""","""Hidup Katolik Magazine""","""National""","""Suatu ketika di masa itu.""",25,"""https://www.hidupkatolik.com/2…"
"""Catholic""","""Dokpen KWI""","""National""","""Dan apa impian Tuhan itu?""",25,"""https://www.dokpenkwi.org/pesa…"
"""Catholic""","""Katolikana""","""National""","""Ini masa yang cukup lama.""",25,"""https://www.katolikana.com/202…"
"""Catholic""","""KWI - Konferensi Waligereja In…","""National""","""Ia tampak muda dan kecil.""",25,"""https://www.mirifica.net/santo…"
"""Catholic""","""KWI - Konferensi Waligereja In…","""National""","""U : Engkau tetap perawan.""",25,"""https://www.mirifica.net/20-no…"
"""Islam""","""Bersama Dakwah""","""National""","""Terdiri dari banyak sub .""",25,"""https://bersamadakwah.net/cate…"
"""Islam""","""Bersama Dakwah""","""National""","""Kembali ke kasus di atas.""",25,"""https://bersamadakwah.net/kont…"


In [31]:
longest_texts = (
    lf
    .with_columns(
        pl.col("Sentence_Unit")
        .str.len_chars()
        .alias("char_length")
    )
    .select(
        [
            "Label",
            "Denomination",
            "Location",
            "Sentence_Unit",
            "char_length",
            "Link",
        ]
    )
    .sort("char_length", descending=True)
    .head(10)
    .collect(engine="streaming")
)

longest_texts

Label,Denomination,Location,Sentence_Unit,char_length,Link
str,str,str,str,u32,str
"""Islam""","""NU Online""","""National""","""Jika diringkas, penjelasan di …",500,"""https://islam.nu.or.id/nikah-k…"
"""Catholic""","""KAS - Keuskupan Agung Semarang""","""Java""","""Para Uskup harus melakukan seg…",500,"""https://kas.or.id/antiquum-min…"
"""Catholic""","""Dokpen KWI""","""National""","""Tahapan pertumbuhan yang diusu…",500,"""https://www.dokpenkwi.org/?p=1…"
"""Protestant""","""Majalah Gaharu""","""National""","""D yang menyatakan bahwa perkem…",500,"""https://www.majalahgaharu.com/…"
"""Protestant""","""PGI - Persekutuan Gereja-gerej…","""National""","""Akhirnya, saya ingin mengutip …",500,"""https://pgi.or.id/news/opini/k…"
"""Islam""","""Muslim.or.id""","""National""","""Sebagaimana firman Ta (yang ar…",500,"""https://buletin.muslim.or.id/k…"
"""Islam""","""DMI - Dewan Masjid Indonesia""","""National""","""Pekerjaan pertama Syekh Wahbah…",500,"""https://dmi.or.id/innalillahi-…"
"""Catholic""","""Dokpen KWI""","""National""","""Para Uskup harus melakukan seg…",500,"""https://www.dokpenkwi.org/anti…"
"""Catholic""","""Katolisitas""","""National""","""Pada saat Rasul Paulus memperi…",500,"""https://katolisitas.org/tak-pe…"


In [32]:
word_length_stats = (
    lf
    .with_columns(
        pl.col("Sentence_Unit")
        .str.replace_all(r"\s+", " ")
        .str.strip_chars()
        .str.split(" ")
        .list.len()
        .alias("word_count")
    )
    .select(
        [
            pl.len().alias("total_rows"),
            pl.col("word_count").mean().alias("mean_words"),
            pl.col("word_count").median().alias("median_words"),
            pl.col("word_count").min().alias("min_words"),
            pl.col("word_count").max().alias("max_words"),
        ]
    )
    .collect(engine="streaming")
)

word_length_stats

total_rows,mean_words,median_words,min_words,max_words
u32,f64,f64,u32,u32
3013172,17.083286,15.0,5,93


In [33]:
text_profile_by_label = (
    lf
    .with_columns(
        [
            pl.col("Sentence_Unit")
            .str.len_chars()
            .alias("char_length"),

            pl.col("Sentence_Unit")
            .str.replace_all(r"\s+", " ")
            .str.strip_chars()
            .str.split(" ")
            .list.len()
            .alias("word_count")
        ]
    )
    .group_by("Label")
    .agg(
        [
            pl.len().alias("jumlah_data"),
            pl.col("char_length").mean().round(2).alias("rata_rata_karakter"),
            pl.col("char_length").median().alias("median_karakter"),
            pl.col("char_length").min().alias("min_karakter"),
            pl.col("char_length").max().alias("max_karakter"),
            pl.col("word_count").mean().round(2).alias("rata_rata_kata"),
            pl.col("word_count").median().alias("median_kata")
        ]
    )
    .sort("jumlah_data", descending=True)
    .collect(engine="streaming")
)

text_profile_by_label

Label,jumlah_data,rata_rata_karakter,median_karakter,min_karakter,max_karakter,rata_rata_kata,median_kata
str,u32,f64,f64,u32,u32,f64,f64
"""Islam""",1455454,124.28,111.0,25,500,17.21,15.0
"""Catholic""",797131,127.27,111.0,25,500,17.72,16.0
"""Protestant""",760587,114.01,99.0,25,500,16.17,14.0


In [34]:
location_counts = (
    lf
    .group_by("Location")
    .agg(
        pl.len().alias("jumlah")
    )
    .sort("jumlah", descending=True)
    .collect(engine="streaming")
)

location_counts

Location,jumlah
str,u32
"""National""",2492876
"""Java""",286452
"""Sumatra""",144632
"""Sulawesi""",30024
"""Nusa Tenggara""",23684
"""Papua""",19199
"""International""",9369
"""Maluku""",5988
"""Kalimantan""",948


In [35]:
location_distribution = (
    location_counts
    .with_columns(
        (
            pl.col("jumlah") / pl.col("jumlah").sum() * 100
        )
        .round(2)
        .alias("persentase")
    )
)

location_distribution

Location,jumlah,persentase
str,u32,f64
"""National""",2492876,82.73
"""Java""",286452,9.51
"""Sumatra""",144632,4.8
"""Sulawesi""",30024,1.0
"""Nusa Tenggara""",23684,0.79
"""Papua""",19199,0.64
"""International""",9369,0.31
"""Maluku""",5988,0.2
"""Kalimantan""",948,0.03


In [36]:
domain_counts = (
    lf
    .select(
        pl.col("Link")
        .str.extract(r"^(?:https?://)?(?:www\.)?([^/]+)", 1)
        .str.to_lowercase()
        .alias("domain")
    )
    .filter(
        pl.col("domain").is_not_null()
        & (pl.col("domain") != "")
    )
    .group_by("domain")
    .agg(
        pl.len().alias("jumlah")
    )
    .sort("jumlah", descending=True)
    .head(20)
    .collect(engine="streaming")
)

domain_counts

domain,jumlah
str,u32
"""ibtimes.id""",147797
"""muslim.or.id""",142354
"""narwastu.id""",133793
"""katolisitas.org""",129996
"""mirifica.net""",129803
…,…
"""gkjw.or.id""",76038
"""majalahgaharu.com""",76018
"""persis.or.id""",71529


In [37]:
top_words = (
    lf
    .select(
        pl.col("Sentence_Unit")
        .str.to_lowercase()
        .str.replace_all(r"[^\p{L}\p{N}]+", " ")
        .str.strip_chars()
        .str.split(" ")
        .alias("words")
    )
    .explode("words")
    .filter(
        (pl.col("words") != "")
        & (pl.col("words").str.len_chars() > 1)
    )
    .group_by("words")
    .agg(
        pl.len().alias("jumlah")
    )
    .sort("jumlah", descending=True)
    .head(20)
    .collect(engine="streaming")
)

top_words

/tmp/ipykernel_315/2805491110.py:11: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("words")


words,jumlah
str,u32
"""yang""",1844278
"""dan""",1541501
"""di""",787175
"""dalam""",638429
"""dengan""",617803
…,…
"""akan""",265417
"""kepada""",261041
"""menjadi""",236748


In [38]:
import duckdb

print("DuckDB version:", duckdb.__version__)

DuckDB version: 1.5.6


In [39]:
duckdb_summary = duckdb.sql("""
    SUMMARIZE
    SELECT *
    FROM read_csv_auto(
        '../data/raw/data.csv',
        header = true
    )
""")

duckdb_summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────┬─────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────┬───────┬───────┬───────┬───────┬───────┬─────────┬─────────────────┐
│  column_name  │ column_type │                                                          min                                                           │                                                                                                                                 

In [40]:
duckdb_count = duckdb.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT Label) AS unique_label,
        COUNT(DISTINCT Denomination) AS unique_denomination,
        COUNT(DISTINCT Location) AS unique_location,
        COUNT(DISTINCT Sentence_Unit) AS unique_sentence,
        COUNT(DISTINCT Link) AS unique_link
    FROM read_csv_auto(
        '../data/raw/data.csv',
        header = true
    )
""")

duckdb_count

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬──────────────┬─────────────────────┬─────────────────┬─────────────────┬─────────────┐
│ total_rows │ unique_label │ unique_denomination │ unique_location │ unique_sentence │ unique_link │
│   int64    │    int64     │        int64        │      int64      │      int64      │    int64    │
├────────────┼──────────────┼─────────────────────┼─────────────────┼─────────────────┼─────────────┤
│    3013172 │            3 │                  60 │               9 │         1966151 │      132075 │
└────────────┴──────────────┴─────────────────────┴─────────────────┴─────────────────┴─────────────┘

In [41]:
duckdb_label = duckdb.sql("""
    SELECT
        Label,
        COUNT(*) AS jumlah,
        ROUND(
            COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
            2
        ) AS persentase
    FROM read_csv_auto(
        '../data/raw/data.csv',
        header = true
    )
    GROUP BY Label
    ORDER BY jumlah DESC
""")

duckdb_label

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────┬────────────┐
│   Label    │ jumlah  │ persentase │
│  varchar   │  int64  │   double   │
├────────────┼─────────┼────────────┤
│ Islam      │ 1455454 │       48.3 │
│ Catholic   │  797131 │      26.45 │
│ Protestant │  760587 │      25.24 │
└────────────┴─────────┴────────────┘

In [42]:
duckdb_text = duckdb.sql("""
    SELECT
        COUNT(*) AS total_rows,
        ROUND(AVG(LENGTH(Sentence_Unit)), 2) AS rata_rata_karakter,
        MEDIAN(LENGTH(Sentence_Unit)) AS median_karakter,
        MIN(LENGTH(Sentence_Unit)) AS min_karakter,
        MAX(LENGTH(Sentence_Unit)) AS max_karakter
    FROM read_csv_auto(
        '../data/raw/data.csv',
        header = true
    )
""")

duckdb_text

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┬─────────────────┬──────────────┬──────────────┐
│ total_rows │ rata_rata_karakter │ median_karakter │ min_karakter │ max_karakter │
│   int64    │       double       │     double      │    int64     │    int64     │
├────────────┼────────────────────┼─────────────────┼──────────────┼──────────────┤
│    3013172 │             122.48 │           108.0 │           25 │          500 │
└────────────┴────────────────────┴─────────────────┴──────────────┴──────────────┘

In [43]:
duckdb_location = duckdb.sql("""
    SELECT
        Location,
        COUNT(*) AS jumlah,
        ROUND(
            COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
            2
        ) AS persentase
    FROM read_csv_auto(
        '../data/raw/data.csv',
        header = true
    )
    GROUP BY Location
    ORDER BY jumlah DESC
""")

duckdb_location

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────┬─────────┬────────────┐
│   Location    │ jumlah  │ persentase │
│    varchar    │  int64  │   double   │
├───────────────┼─────────┼────────────┤
│ National      │ 2492876 │      82.73 │
│ Java          │  286452 │       9.51 │
│ Sumatra       │  144632 │        4.8 │
│ Sulawesi      │   30024 │        1.0 │
│ Nusa Tenggara │   23684 │       0.79 │
│ Papua         │   19199 │       0.64 │
│ International │    9369 │       0.31 │
│ Maluku        │    5988 │        0.2 │
│ Kalimantan    │     948 │       0.03 │
└───────────────┴─────────┴────────────┘

# Kesimpulan Data Profiling

Berdasarkan hasil profiling menggunakan Polars dan DuckDB, dataset Indonesian Religious Corpus yang digunakan memiliki ukuran file sekitar 682,73 MiB dengan total 3.013.172 baris dan 7 kolom.

Dataset terdiri dari tiga label utama yaitu Islam, Catholic, dan Protestant. Distribusi label menunjukkan bahwa Islam merupakan kategori terbesar dengan 1.455.454 data (48,30%), diikuti Catholic sebanyak 797.131 data (26,45%), dan Protestant sebanyak 760.587 data (25,24%).

Dataset memiliki 60 kategori Denomination dan 9 kategori Location. Location paling dominan adalah National dengan 2.492.876 data atau sekitar 82,73%, sedangkan Java berada di posisi kedua dengan 286.452 data atau 9,51%.

Pada pemeriksaan missing values, kolom Label, Denomination, Location, Sentence_Unit, dan Link tidak memiliki nilai NULL. Sebaliknya, kolom Date dan Title memiliki NULL sebesar 100%, sehingga kedua kolom tersebut tidak dapat digunakan secara langsung untuk analisis temporal maupun analisis berdasarkan judul.

Hasil pemeriksaan duplikasi menunjukkan terdapat 355 duplicate rows dari total 3.013.172 baris atau sekitar 0,0118%. Selain itu, ditemukan adanya pengulangan Sentence_Unit sehingga diperlukan perhatian khusus pada tahap data cleaning.

Kolom Sentence_Unit memiliki rata-rata panjang sekitar 122,48 karakter, median 108 karakter, panjang minimum 25 karakter, dan maksimum 500 karakter. Rata-rata jumlah kata adalah sekitar 17,08 kata dengan median 15 kata.

Distribusi label menunjukkan dataset relatif tidak seimbang, tetapi belum termasuk sangat ekstrem. Kategori terbesar yaitu Islam memiliki jumlah data sekitar 1,91 kali kategori terkecil yaitu Protestant.

Dataset memenuhi kriteria tugas Analisis Big Data karena memiliki lebih dari 3 juta record dan ukuran file lebih dari 500 MB. Penggunaan Polars dengan lazy evaluation dan DuckDB membantu melakukan profiling serta analisis agregasi terhadap dataset berukuran besar secara lebih efisien.